# Week 4 · Checkpoint + Resume + TensorBoard

> **本周一句话**:训练随时能断、断了能续、断之前的曲线能看 —— 把训练代码从"实验室一次性"变成"工程闭环"。

本周不会有新的 val loss 突破。但学完之后,你的 v0.7 就可以放心训 8 小时 + 中间断电 + 重启续训 + 在浏览器看实时曲线。**为 Week 5 25M 大模型做完整准备**。

## 0. 本周目标

| 版本 | 新增 | val | 工程能力 |
|---|---|---|---|
| v0.5(上周) | AMP + clip + wd | 4.21 | 训完只剩 model.state_dict() |
| v0.6 | + 完整 checkpoint | 4.21 | best.pt + latest.pt,可恢复全部状态 |
| v0.7 | + Resume + TensorBoard | 4.21 | 续训不重头,实时监控 |

关键产出:
- `checkpoints/v06/{best,latest}.pt` —— 完整训练状态打包
- `tb_logs/v07_*/` —— TensorBoard event 文件
- 可以 `python train/train_v07.py --resume` 续训

## 1. 前置知识

**必备**:
- Week 3 跑完(知道 AMP / GradScaler 怎么用)
- 知道 `torch.save / torch.load`

**这周第一次遇到**:
- `state_dict()` 包含什么(model / optimizer / scaler 各自的)
- `weights_only=True/False`(PyTorch 2.4+ 的安全机制)
- TensorBoard `SummaryWriter`、`add_scalar`

## 2. 核心概念

### 2.1 一个完整的 checkpoint 要存什么

只存 `model.state_dict()` 是**不够的**。完整 checkpoint 需要:

```python
{
    "step":             3500,                  # 走到第几步
    "model_state":      model.state_dict(),    # 模型权重
    "optimizer_state":  optimizer.state_dict(),# Adam 的 m / v 动量
    "scaler_state":     scaler.state_dict(),   # AMP scale 当前值
    "best_val":         4.21,                  # 历史最佳,resume 后接着比
    "config":           {n_embed: 192, ...},   # 模型结构,重建用
}
```

**为什么 optimizer 必须存**?Adam 维护着每个参数的两个动量(`m` 一阶、`v` 二阶)。这些动量本身就是训练历史的"记忆" —— 不存的话 resume 等于从头初始化 optimizer,前 500 步训练全废。

**为什么 config 要存**?推理 / 评测时要重建模型,如果忘了当时用的 `n_embed=192` 还是 384,载入权重时会 shape mismatch。

### 2.2 best.pt vs latest.pt 的双轨保存

```
       ┌─────────────────────┐
       │  best.pt            │  ← 每次 val 创新低就覆盖
       │  (推理 / 评测用)   │
       └─────────────────────┘

       ┌─────────────────────┐
       │  latest.pt          │  ← 每 1000 步覆盖一次
       │  (resume 续训用)   │
       └─────────────────────┘
```

为什么要两个?
- **best**:跑评测、生成、Week 5 之后做 fine-tune 起点 —— 要"模型最好的瞬间"
- **latest**:断点续训 —— 要"最近的瞬间",哪怕 val 当时刚好在波动期

一个文件兼顾两用 → 你要不要用 best 又开始训(把最佳状态训坏),要不要用最近又拿不到最佳 —— 两难。**用两个文件,问题消失**。

### 2.3 Resume 时 LR Schedule 怎么对齐

上周 v0.4/v0.5 的 LR schedule 是按 `step` 索引的:

```python
lr = warmup_cosine(step, warmup=200, total=8000, ...)
```

如果从 step 3500 续训到 8000(只剩 4500 步),硬接 schedule 会很尴尬:warmup 早过了,cosine 也已经降到一半 —— 续训这段几乎用最小 lr,基本学不到东西。

**我们的做法**(`train_v07.py --resume`):

- 重新定义一个**小 schedule**:max_lr=1e-4(原 3e-4 的 1/3)、warmup=100、total=4000
- 让续训阶段有自己的"暖身 + 衰减"
- 全局 `step` 继续递增(`start_step + local_step`),用于 TensorBoard 横轴和保存

本质思想:**resume = 在已有权重基础上开一段新训练**,不是机械接续。

### 2.4 TensorBoard 监控什么有用

不要无脑写所有东西。写少而精的指标:

| 指标 | 写入频率 | 看什么 |
|---|---|---|
| `lr` | 每步 | 确认 schedule 形状对 |
| `train/loss_step` | 每步 | 单步噪声大,但能看到瞬时尖刺 |
| `grad_norm` | 每步 | **诊断利器** —— 突然飙升 = 数据/初始化有问题 |
| `train/loss_eval` | 每 eval_interval | 平滑曲线,看趋势 |
| `val/loss` | 每 eval_interval | 关键指标 |

看 `grad_norm` 比看 loss 更早发现问题 —— loss 还没爆的时候 grad_norm 已经飙到 10+。我们的 v0.7 把 `grad_norm` 当一等公民。

## 3. 代码地图

| 文件 | 行数 | 干什么 |
|---|---|---|
| `train/checkpoint.py` | 50 | `save_checkpoint` / `load_checkpoint`,接口干净 |
| `train/train_v06.py` | 130 | v0.5 + 双轨 checkpoint |
| `train/train_v07.py` | 165 | v0.6 + `--resume` + TensorBoard |

`checkpoint.py` 是工具库,被 v06 / v07 / Week 5 v08 / Week 6 SFT / Week 7 DPO **复用**。一个干净的小工具能消除很多重复代码。

## 4. 动手做

In [ ]:
import subprocess, sys


def run(cmd):
    """跑子进程并把输出实时打印到 cell。
    subprocess.run 默认把子进程 stdout 写到 kernel 原始 fd, notebook 看不到;
    用 Popen 逐行回读才能在 cell 里实时看到脚本的 print。
    "python" 换成 sys.executable, 确保用当前 kernel 解释器。"""
    cmd = [sys.executable if c == "python" else c for c in cmd]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"{cmd} 退出码 {proc.returncode}")

# v0.6: 训练 + 保存 best/latest
run(["python", "../train/train_v06.py"])
# 预期: best.pt 和 latest.pt 各约 80 MB(模型 + 优化器状态)

In [ ]:
import os
ckpt_dir = "../checkpoints/v06"
for fn in os.listdir(ckpt_dir):
    size = os.path.getsize(f"{ckpt_dir}/{fn}") / 1e6
    print(f"  {fn}: {size:.2f} MB")

In [ ]:
# v0.7: 从零训(带 TensorBoard)
run(["python", "../train/train_v07.py"])
# tb_logs/v07_fresh_*/ 下会有 event 文件

In [ ]:
# v0.7: 续训测试(从 v0.6 的 latest 接着训 4000 步)
run(["python", "../train/train_v07.py", "--resume"])
# 预期: 一开始 val loss 接近 4.21(说明 resume 成功),
#       而不是 9.2(说明 model_state 没加载上)

**启动 TensorBoard**:

```powershell
tensorboard --logdir tb_logs
```

浏览器开 http://localhost:6006,选 `v07_*` 的几个 run 对比。**会看到**:
- `lr` 是 warmup + cosine 的标准形状
- `train/loss_step` 抖动很大(单步噪声),但整体下行
- `train/loss_eval` 是平滑曲线(20 个 batch 平均)
- `val/loss` 平滑下降
- `grad_norm` 一般在 0.3~2 之间,偶尔尖刺到 5+,被 clip 救回

## 5. 自测题

**A. Checkpoint**
- A1 如果只存 `model.state_dict()` 续训,前 1000 步会和正常训练有什么差别?为什么?
- A2 `latest.pt` 每 1000 步覆盖一次,如果在第 999 步崩了,丢了多少?如果改成每 100 步覆盖,代价是什么?
- A3 `weights_only=True`(默认)和 `weights_only=False` 各有什么风险?为什么我们的 `load_checkpoint` 用 `False`?

**B. Resume**
- B1 resume 时如果用原 schedule(max_lr=3e-4 / warmup=200 / total=8000)继续,从 step 3500 跑到 8000,会发生什么?
- B2 我们的 resume 用 max_lr=1e-4 比原 3e-4 小 3 倍,这个比例怎么定的?如果还用 3e-4 会怎样?
- B3 如果训练时 model 结构变了(比如加了一层),还能 `load_state_dict` 吗?

**C. TensorBoard**
- C1 为什么 `train/loss_step` 抖动很大,`train/loss_eval` 却很平滑?
- C2 `grad_norm` 突然飙到 10+,但 loss 还没崩 —— 你会怎么处理?
- C3 多次 resume 同一 ckpt 会在 TB 里产生多条线吗?怎么对比?

## 6. 容易踩的坑

**坑 1:`torch.load(weights_only=True)` 加载完整 ckpt 会失败**

PyTorch 2.4+ 默认 `weights_only=True`,只接受纯权重(`Tensor`),拒绝完整 dict + Python 对象。我们的 ckpt 含 config dict,必须 `weights_only=False`。

**坑 2:resume 前忘了重建 optimizer / scaler**

```python
# 错
optimizer = ...  # 已经存在
optimizer.load_state_dict(ckpt["optimizer_state"])   # 报错: param shape mismatch

# 对
model = MyModel(**config); model.to(device)
optimizer = AdamW(model.parameters(), ...)            # ← 必须用新模型的 parameters
optimizer.load_state_dict(ckpt["optimizer_state"])
```

optimizer 状态和 model.parameters() 一一对应,顺序不能错。

**坑 3:多个 run 写到同一 TB 目录 → 曲线混在一起**

我们用 `f"v07_{tag}_{datetime.now().strftime(...)}"` 给每个 run 唯一目录。同名会被 TB 当作同一个 run 续写,曲线乱跳。

**坑 4:resume 后 TB 横轴从 0 重新开始**

我们传给 `add_scalar` 的是 `global_step = start_step + local_step`,不是 local_step。横轴连续才能看到完整曲线。

**坑 5:checkpoint 越存越多撑爆磁盘**

Week 5 v08 / Week 6 SFT 的 checkpoint 各 ~100 MB,跑几个版本就吃 GB。HuggingFace trainer 有 `save_total_limit`,我们的 `train_loop` 没有 —— 自己定期 `rm` 老的。

## 7. 进入 Week 5 前

现在你应该:
- ☑ `checkpoints/v06/{best,latest}.pt` 都存在,各 ~80 MB
- ☑ 跑过 `--resume`,看到续训从 4.21 附近开始(不是 9.2)
- ☑ TensorBoard 里能看到 lr / loss / grad_norm 三条核心曲线
- ☑ 能解释 best vs latest 为什么必须分开

Week 5 数据规模 ×5、模型规模 ×4 —— 唐+宋诗+宋词,25M 参数。**没有 Week 4 的工程能力,Week 5 训不完**(中间一次电脑断电就全废)。